# 2 · Leitura de faixas e integração

Este notebook aplica o detector treinado no notebook 1 e estuda leitura de faixas em imagens rotuladas do Barrettotte, conjunto complementar necessário porque o Samu não tem rótulos de valor. As caixas de corpo não foram anotadas: a imagem inteira é a entrada do benchmark. São apresentados os erros e as leituras recusadas. A concordância entre regras é uma **heurística**, não uma probabilidade calibrada.

As referências MATLAB e Automatic Component Analyzer são implementações compatíveis disponíveis no repositório; a versão original MATLAB e a reprodução anterior são documentadas em `benchmarks/EXECUTION_REPORT.md`.

In [ ]:
from pathlib import Path
import os, sys, json, hashlib, time
from collections import Counter, defaultdict
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
from ultralytics import YOLO
def read_image(path):
    try: return cv2.imdecode(np.fromfile(str(path),dtype=np.uint8),cv2.IMREAD_COLOR)
    except (OSError,cv2.error): return None

PROJECT = Path(os.environ.get("RESISTOR_PROJECT_ROOT", "..")).expanduser().resolve()
DATA = Path(os.environ.get("RESISTOR_DATA_ROOT", str(PROJECT / "data"))).expanduser().resolve()
OUTPUT = Path(os.environ.get("RESISTOR_OUTPUT", str(PROJECT / "outputs/proposta"))).expanduser().resolve()
sys.path.insert(0, str(PROJECT / "src"))
from resistor_detector.colors import classify_bgr, classify_bgr_lab_pca, decode_color_bands
from resistor_detector.detection import sample_band_colors
from resistor_detector.matlab_compat import analyze_matlab_compatible
from resistor_detector.automatic_component_analyzer import analyze_frame as analyzer

manifest_path = PROJECT / "data/manifests/barrettotte-crop-value.jsonl"
assert manifest_path.exists(), f"Manifesto não encontrado: {manifest_path}"
OUTPUT.mkdir(parents=True, exist_ok=True)
threshold_path=OUTPUT / "limiar_escolhido_validacao.json"
threshold=float(os.environ.get("RESISTOR_CONFIDENCE", json.loads(threshold_path.read_text())["limiar"] if threshold_path.exists() else .25))
print("Dados:", DATA, "\nResultados:", OUTPUT)


## Recorte e alinhamento pelo eixo

É estimado o primeiro eixo principal do primeiro plano em CIELAB, seguido de rotação. O resultado é uma aproximação: recortes cujo fundo dificulte a segmentação devem ser revisados. Não se assume que todos os resistores de uma imagem tenham o mesmo valor.

In [ ]:
def align_crop(image):
    h,w = image.shape[:2]
    if h < 8 or w < 8: return image, False
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB).astype(np.float32)
    border = np.concatenate([lab[0,:,:],lab[-1,:,:],lab[:,0,:],lab[:,-1,:]],axis=0)
    bg = np.median(border,axis=0)
    dist = np.linalg.norm(lab-bg,axis=2)
    mask = (dist > 25).astype(np.uint8)*255
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3,3),np.uint8))
    num, labels, stats, _ = cv2.connectedComponentsWithStats(mask)
    if num < 2: return image, False
    label = 1 + np.argmax(stats[1:,cv2.CC_STAT_AREA])
    ys,xs = np.where(labels==label)
    if len(xs)<100: return image, False
    pts = np.column_stack((xs,ys)).astype(np.float32)
    _, vectors = cv2.PCACompute(pts, mean=None, maxComponents=2)
    angle = np.degrees(np.arctan2(vectors[0,1],vectors[0,0]))
    if angle > 90: angle -= 180
    if angle < -90: angle += 180
    center=(w/2,h/2)
    matrix=cv2.getRotationMatrix2D(center, float(angle), 1.0)
    rotated=cv2.warpAffine(image,matrix,(w,h),flags=cv2.INTER_LINEAR,borderMode=cv2.BORDER_REPLICATE)
    rotated_mask=cv2.warpAffine((labels==label).astype(np.uint8),matrix,(w,h),flags=cv2.INTER_NEAREST)
    ys,xs=np.where(rotated_mask>0)
    if len(xs)==0: return image, False
    x0,x1=max(0,int(xs.min())),min(w,int(xs.max())+1)
    y0,y1=max(0,int(ys.min())),min(h,int(ys.max())+1)
    return rotated[y0:y1,x0:x1], True


## Perfis de cor e decisão conservadora

Para quatro e cinco faixas, o perfil central é classificado pelas regras HSV e CIELAB/PCA já estudadas. Ambas as orientações são consideradas. A sequência só é apresentada automaticamente quando os classificadores concordam em uma única interpretação válida. Quando houver alternativas, a saída solicita revisão. Este protótipo ainda usa posições aproximadas e sua qualidade é medida abaixo.

In [ ]:
def read_bands(image):
    aligned, aligned_ok = align_crop(image)
    if aligned.size == 0 or not aligned_ok: return {"status":"revisar","ohms":None,"tolerancia":None,"faixas":[],"motivo":"eixo não identificado"}
    h,w=aligned.shape[:2]
    candidates = []
    for n in (4,5):
        proposals = []
        for classifier in (classify_bgr, classify_bgr_lab_pca):
            colors=sample_band_colors(aligned,(0,0,w,h),band_count=n,classifier=classifier)
            possibilities=[]
            for order in (colors,tuple(reversed(colors))):
                if any(color is None for color in order): continue
                decoded=decode_color_bands(order)
                if decoded is not None and order[0] != "black":
                    possibilities.append((tuple(order),float(decoded.ohms),float(decoded.tolerance_percent)))
            proposals.append(set(possibilities))
        candidates.extend(proposals[0].intersection(proposals[1]))
    unique=list(set(candidates))
    if len(unique)!=1:
        return {"status":"revisar","ohms":None,"tolerancia":None,"faixas":[],"motivo":"faixas ou orientação ambíguas","alternativas":len(unique),"eixo_estimado":aligned_ok}
    bands, ohms, tolerance=unique[0]
    return {"status":"leitura_candidata","ohms":ohms,"tolerancia":tolerance,"faixas":list(bands),"motivo":"concordância de regras; requer validação","eixo_estimado":aligned_ok}


In [ ]:
quatro=decode_color_bands(("yellow","violet","brown","gold"))
cinco=decode_color_bands(("yellow","violet","black","black","brown"))
assert quatro.ohms == 470 and quatro.tolerance_percent == 5
assert cinco.ohms == 470 and cinco.tolerance_percent == 1
assert read_bands(np.zeros((2,2,3),dtype=np.uint8))["status"] == "revisar"
print("Sanidade: 4 e 5 faixas e recorte insuficiente verificados.")


## Benchmark das referências no mesmo conjunto

O teste usa grupos por pasta de classe (valor/tolerância/potência), de modo determinístico. Isso evita colocar arquivos da mesma pasta em treino e teste, mas **não garante** separação de componentes físicos entre pastas. Os rótulos de Barrettotte referem-se à imagem inteira: esta avaliação não mede localização nem usa recorte de corpo anotado. Opcionalmente, `RESISTOR_CROP_WEIGHTS` usa um detector para recortar antes da leitura; faltas na detecção são contabilizadas como leitura recusada. Os tempos de leitura não incluem a detecção. Os sistemas de referência podem retornar mais de uma hipótese; isso é contabilizado como ambíguo.

In [ ]:
rows=[json.loads(line) for line in manifest_path.read_text(encoding="utf-8").splitlines() if line.strip()]
for row in rows:
    relative=Path(row["path"])
    row["_image"] = DATA / relative.relative_to("data")
    row["_group"] = relative.parent.name
# Grupos de pastas em vez de imagens individuais.
groups=sorted(set(r["_group"] for r in rows),key=lambda g:hashlib.sha256(("42:"+g).encode()).hexdigest())
cut1,cut2=int(len(groups)*.70),int(len(groups)*.85)
part={g: ("train" if i<cut1 else "val" if i<cut2 else "test") for i,g in enumerate(groups)}
selected_partition=os.environ.get("RESISTOR_READING_PART", "test")
assert selected_partition in ("train","val","test")
test=[r for r in rows if part[r["_group"]]==selected_partition]
assert test and all(r["_image"].exists() for r in test), "Imagens Barrettotte ausentes; confira RESISTOR_DATA_ROOT."
print("Grupos e imagens:", Counter(part.values()), len(test))


In [ ]:
def reference_matlab(image):
    out=analyze_matlab_compatible(image)
    values=[x["value_ohms"] for x in out["detections"] if x["value_ohms"] is not None]
    return {"ohms":float(values[0]) if len(values)==1 else None,"tolerancia":None,"status":"candidato" if len(values)==1 else "revisar"}
def reference_analyzer(image):
    options=analyzer(image)["decoded_results"]
    choices={(float(x["value"]), str(x["tolerance"])) for x in options}
    if len(choices)!=1: return {"ohms":None,"tolerancia":None,"status":"revisar"}
    value,tol=next(iter(choices))
    try: tol=float(tol.replace("%","").replace("±","").strip())
    except (ValueError,AttributeError): tol=None
    return {"ohms":value,"tolerancia":tol,"status":"candidato"}

# Opcional: usar um detector para obter a região antes de avaliar a leitura.
crop_weights=os.environ.get("RESISTOR_CROP_WEIGHTS")
crop_model=YOLO(crop_weights) if crop_weights else None
records=[]
for row in test:
    image=read_image(row["_image"]); assert image is not None, row["_image"]
    actual=row["objects"][0]
    if crop_model is not None:
        prediction=crop_model.predict(image,conf=.25,device=0 if torch.cuda.is_available() else "cpu",verbose=False)[0]
        if len(prediction.boxes):
            index=int(prediction.boxes.conf.argmax())
            x1,y1,x2,y2=map(int,prediction.boxes.xyxy[index].cpu().numpy())
            image=image[max(0,y1):min(image.shape[0],y2),max(0,x1):min(image.shape[1],x2)]
        else: image=None
    for name,fn in (("perfis_consenso",read_bands),("matlab_compat",reference_matlab),("component_analyzer",reference_analyzer)):
        start=time.perf_counter()
        result=fn(image) if image is not None and image.size else {"ohms":None,"tolerancia":None,"status":"sem deteccao"}
        value=result.get("ohms"); tolerance=result.get("tolerancia")
        truth=float(actual["ohms"]); truth_tol=actual.get("tolerance_percent")
        records.append({"id":row["id"],"metodo":name,"real_ohms":truth,"previsto_ohms":value,"real_tolerancia":truth_tol,"previsto_tolerancia":tolerance,
                        "valor_correto":value is not None and np.isclose(value,truth,rtol=1e-9),
                        "dentro_tolerancia":value is not None and truth_tol is not None and abs(value-truth)<=abs(truth)*float(truth_tol)/100,
                        "tolerancia_correta":tolerance is not None and truth_tol is not None and np.isclose(tolerance,truth_tol),
                        "ms":(time.perf_counter()-start)*1000})
reading=pd.DataFrame(records)
reading.to_csv(OUTPUT / "leitura_por_imagem.csv",index=False)
def reading_summary(g):
    emitted=g.previsto_ohms.notna()
    return pd.Series({"imagens":len(g),"cobertura":float(emitted.mean()),"valor_exato_total":float(g.valor_correto.mean()),
        "valor_exato_emitidos":float(g.loc[emitted,"valor_correto"].mean()) if emitted.any() else np.nan,
        "dentro_tolerancia_total":float(g.dentro_tolerancia.mean()),"tolerancia_exata_total":float(g.tolerancia_correta.mean()),"tempo_medio_ms":g.ms.mean()})
summary=reading.groupby("metodo").apply(reading_summary,include_groups=False)
summary.to_csv(OUTPUT / "leitura_resumo.csv")
display(summary)


## Integração: uma imagem, várias detecções

Execute após o notebook 1. Configure `RESISTOR_IMAGE` para uma imagem de teste própria; o código não usa o valor verdadeiro como entrada. Um valor só é mostrado quando a regra de consenso retorna uma única interpretação; as demais saídas pedem revisão.

In [ ]:
best=OUTPUT / "treino/yolo-samu/weights/best.pt"
image_path=os.environ.get("RESISTOR_IMAGE")
if not best.exists():
    print("Execute primeiro o notebook 1 para treinar e avaliar o detector.")
elif not image_path:
    print("Defina RESISTOR_IMAGE com o caminho de uma imagem estática para visualizar a saída integrada.")
else:
    model=YOLO(str(best))
    frame=read_image(image_path)
    if frame is None: raise FileNotFoundError(image_path)
    start=time.perf_counter()
    prediction=model.predict(frame,conf=threshold,iou=.7,device=0 if torch.cuda.is_available() else "cpu",verbose=False)[0]
    boxed=frame.copy(); result_rows=[]
    items=sorted(zip(prediction.boxes.xyxy.cpu().numpy(),prediction.boxes.conf.cpu().numpy()),key=lambda item:(item[0][1],item[0][0]))
    for n,(box,score) in enumerate(items,1):
        x1,y1,x2,y2=map(int,box)
        x1,y1=max(0,x1),max(0,y1); x2,y2=min(frame.shape[1],x2),min(frame.shape[0],y2)
        result=read_bands(frame[y1:y2,x1:x2])
        result_rows.append({"resistor":f"R{n}","caixa":[x1,y1,x2,y2],"confianca_deteccao":float(score),**result})
        cv2.rectangle(boxed,(x1,y1),(x2,y2),(30,210,80),2)
        cv2.putText(boxed,f"R{n}",(x1,max(14,y1-5)),cv2.FONT_HERSHEY_SIMPLEX,.6,(30,210,80),2)
    print(f"Quantidade: {len(result_rows)} · tempo total: {(time.perf_counter()-start)*1000:.1f} ms")
    display(pd.DataFrame(result_rows))
    plt.figure(figsize=(12,8)); plt.imshow(cv2.cvtColor(boxed,cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()


## Interpretação

As métricas de leitura incluem recusas no denominador. A métrica condicionada às emissões deve ser vista junto com a cobertura. A confiança de localização do YOLO não é confiança do valor. Para avaliar o pipeline completo faltam imagens com caixas e valor/tolerância anotados conjuntamente, separadas por cena e identidade física.